# Домашнее задание 6: Оптимизация Spark DataFrame

**Задание:** Реализовать 3 кейса, где DataFrame гарантированно быстрее RDD. Для каждого кейса привести код и объяснить, какая оптимизация Catalyst/Tungsten дает выигрыш.

**Дополнительно (*):** Найти 2 кейса, где SQL-запрос выполняется быстрее эквивалентного DataFrame API. Сравнить планы выполнения через `explain()` и объяснить причину разницы.

## Датасет
Используется Online Retail Dataset (~1M транзакций продаж)

**Колонки:**
- InvoiceNo - номер заказа
- StockCode - код товара
- Description - описание товара
- Quantity - количество
- InvoiceDate - дата заказа
- UnitPrice - цена за единицу
- CustomerID - ID покупателя
- Country - страна

In [1]:
import os
import time
import findspark

os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"

findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = SparkSession.builder \
    .appName("Spark_HW6_DataFrame_Optimization") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/06 21:07:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1


## 1. Загрузка и подготовка данных

In [2]:
df = spark.read.csv(
    "OnlineRetail.csv",
    header=True,
    inferSchema=True
)

print(f"Количество строк: {df.count():,}")
print("\nСхема данных:")
df.printSchema()
print("\nПервые 5 строк:")
df.show(5, truncate=False)

Количество строк: 5,000,000

Схема данных:
root
 |-- InvoiceNo: integer (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)


Первые 5 строк:
+---------+---------+------------------------------+--------+----------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                   |Quantity|InvoiceDate     |UnitPrice|CustomerID|Country       |
+---------+---------+------------------------------+--------+----------------+---------+----------+--------------+
|553388   |20719    |WOODLAND CHARLOTTE BAG        |9       |04/03/2011 00:31|41.93    |17602     |Belgium       |
|564191   |20727    |LUNCH BAG BLACK SKULL         |23      |06/22/2011 09:07|7.38     |12872     |United Kingdom|
|566491   |22423    |REGEN

In [3]:
df = df.dropna(subset=["CustomerID"])

df = df.withColumn("Revenue", F.col("Quantity") * F.col("UnitPrice"))

df.cache()
print(f"Количество строк после очистки: {df.count():,}")
df.show(5)

Количество строк после очистки: 3,750,065
+---------+---------+--------------------+--------+----------------+---------+----------+--------------+------------------+
|InvoiceNo|StockCode|         Description|Quantity|     InvoiceDate|UnitPrice|CustomerID|       Country|           Revenue|
+---------+---------+--------------------+--------+----------------+---------+----------+--------------+------------------+
|   553388|    20719|WOODLAND CHARLOTT...|       9|04/03/2011 00:31|    41.93|     17602|       Belgium|            377.37|
|   564191|    20727|LUNCH BAG BLACK S...|      23|06/22/2011 09:07|     7.38|     12872|United Kingdom|            169.74|
|   566491|    22423|REGENCY CAKESTAND...|      48|12/09/2010 10:15|     3.85|     14958|   Switzerland|             184.8|
|   544617|    47566|       PARTY BUNTING|      12|10/07/2011 07:23|    25.79|     13823|      Portugal|            309.48|
|   569580|    22457|NATURAL SLATE HEA...|      38|09/28/2011 04:44|    38.19|     12963| 

## КЕЙС 1: Множественные агрегации

**Задача:** Вычислить для каждой страны:
- Общую выручку (SUM)
- Среднюю выручку (AVG)
- Минимальную выручку (MIN)
- Максимальную выручку (MAX)
- Количество транзакций (COUNT)

**Почему DataFrame быстрее:**
1. **Catalyst Optimizer** - объединяет все агрегации в один проход по данным
2. **Tungsten** - использует бинарный формат без сериализации в объекты
3. **Whole-stage code generation** - генерирует оптимизированный байт-код для всех агрегаций сразу

В RDD пришлось бы делать несколько проходов или писать сложную логику вручную.

In [4]:
print("=" * 80)
print("КЕЙС 1: Множественные агрегации - DataFrame")
print("=" * 80)

start_time = time.time()

result_df = df.groupBy("Country").agg(
    F.sum("Revenue").alias("TotalRevenue"),
    F.avg("Revenue").alias("AvgRevenue"),
    F.min("Revenue").alias("MinRevenue"),
    F.max("Revenue").alias("MaxRevenue"),
    F.count("*").alias("TransactionCount")
).orderBy(F.desc("TotalRevenue"))

result_df.show(10, truncate=False)

df_time = time.time() - start_time
print(f"\n⏱️  Время выполнения DataFrame: {df_time:.3f} сек")

print("\n📊 План выполнения DataFrame:")
result_df.explain(mode="formatted")

КЕЙС 1: Множественные агрегации - DataFrame
+--------------+--------------------+-----------------+----------+----------+----------------+
|Country       |TotalRevenue        |AvgRevenue       |MinRevenue|MaxRevenue|TransactionCount|
+--------------+--------------------+-----------------+----------+----------+----------------+
|Belgium       |1.8658195065000024E8|644.3481151167096|0.5       |2499.5    |289567          |
|Spain         |1.864707970500003E8 |645.3482555564026|0.51      |2500.0    |288946          |
|Portugal      |1.8627106749000013E8|644.7663613329322|0.5       |2499.5    |288897          |
|Italy         |1.8594898179999995E8|644.0862266281492|0.5       |2499.5    |288702          |
|Austria       |1.8578270333999982E8|644.6445912822606|0.5       |2499.5    |288194          |
|France        |1.8572960912999973E8|643.2483856589205|0.51      |2500.0    |288737          |
|Denmark       |1.8563956083000004E8|645.2203077003286|0.51      |2499.5    |287715          |
|Unite

In [5]:
print("=" * 80)
print("КЕЙС 1: Множественные агрегации - RDD (для сравнения)")
print("=" * 80)

start_time = time.time()

rdd = df.rdd.map(lambda row: (row.Country, row.Revenue))

sum_rdd = rdd.reduceByKey(lambda a, b: a + b)
count_rdd = rdd.mapValues(lambda x: (x, 1)).reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
min_rdd = rdd.reduceByKey(lambda a, b: min(a, b))
max_rdd = rdd.reduceByKey(lambda a, b: max(a, b))

sum_dict = sum_rdd.collectAsMap()
count_dict = count_rdd.collectAsMap()
min_dict = min_rdd.collectAsMap()
max_dict = max_rdd.collectAsMap()

result_rdd = []
for country in sum_dict.keys():
    total = sum_dict[country]
    cnt = count_dict[country][1]
    avg = count_dict[country][0] / cnt
    min_val = min_dict[country]
    max_val = max_dict[country]
    result_rdd.append((country, total, avg, min_val, max_val, cnt))

result_rdd.sort(key=lambda x: x[1], reverse=True)

rdd_time = time.time() - start_time

print("\nТоп-10 стран по выручке:")
for i, (country, total, avg, min_val, max_val, cnt) in enumerate(result_rdd[:10], 1):
    print(f"{i}. {country}: Total={total:.2f}, Avg={avg:.2f}, Min={min_val:.2f}, Max={max_val:.2f}, Count={cnt}")

print(f"\n⏱️  Время выполнения RDD: {rdd_time:.3f} сек")
print(f"🚀 DataFrame быстрее в {rdd_time/df_time:.2f}x раз!")

КЕЙС 1: Множественные агрегации - RDD (для сравнения)


[Stage 20:>                                                       (0 + 14) / 14]


Топ-10 стран по выручке:
1. Belgium: Total=186581950.65, Avg=644.35, Min=0.50, Max=2499.50, Count=289567
2. Spain: Total=186470797.05, Avg=645.35, Min=0.51, Max=2500.00, Count=288946
3. Portugal: Total=186271067.49, Avg=644.77, Min=0.50, Max=2499.50, Count=288897
4. Italy: Total=185948981.80, Avg=644.09, Min=0.50, Max=2499.50, Count=288702
5. Austria: Total=185782703.34, Avg=644.64, Min=0.50, Max=2499.50, Count=288194
6. France: Total=185729609.13, Avg=643.25, Min=0.51, Max=2500.00, Count=288737
7. Denmark: Total=185639560.83, Avg=645.22, Min=0.51, Max=2499.50, Count=287715
8. United Kingdom: Total=185625596.37, Avg=643.23, Min=0.52, Max=2499.50, Count=288585
9. Switzerland: Total=185576510.46, Avg=643.53, Min=0.52, Max=2498.50, Count=288372
10. Netherlands: Total=185504342.39, Avg=643.11, Min=0.51, Max=2499.50, Count=288450

⏱️  Время выполнения RDD: 5.284 сек
🚀 DataFrame быстрее в 13.77x раз!


### Объяснение оптимизаций в Кейсе 1

**Catalyst Optimizer:**
- Объединяет все 5 агрегаций в один проход по данным
- Создает единый план выполнения вместо 5 отдельных операций
- Оптимизирует порядок операций (сначала фильтрация, потом агрегация)

**Tungsten:**
- Данные хранятся в бинарном формате (off-heap memory)
- Нет сериализации/десериализации в объекты Java
- Меньше нагрузка на Garbage Collector

**Whole-stage code generation:**
- Генерирует оптимизированный байт-код для всего pipeline
- Устраняет виртуальные вызовы функций
- Использует CPU cache эффективнее

**RDD недостатки:**
- Требует несколько проходов по данным (4 reduceByKey)
- Сериализация данных на каждом шаге
- Нет автоматической оптимизации
- Ручное объединение результатов через collectAsMap()

## КЕЙС 2: JOIN с broadcast оптимизацией

**Задача:** Найти все транзакции покупателей из топ-5 стран по выручке. Это требует JOIN большой таблицы с маленькой.

**Почему DataFrame быстрее:**
1. **Catalyst автоматически применяет broadcast join** - маленькая таблица рассылается всем executor'ам
2. **Нет shuffle** - данные не перемещаются по сети
3. **Tungsten** - эффективная работа с памятью при JOIN
4. **Predicate pushdown** - фильтры применяются до JOIN

В RDD пришлось бы:
- Вручную делать broadcast или обычный join (с shuffle)
- Множественные проходы по данным
- Сериализация при передаче данных
- Нет автоматической оптимизации

In [6]:
print("=" * 80)
print("КЕЙС 2: JOIN с broadcast - DataFrame")
print("=" * 80)

start_time = time.time()

top_countries = df.groupBy("Country").agg(
    F.sum("Revenue").alias("TotalRevenue")
).orderBy(F.desc("TotalRevenue")).limit(5)

print("Топ-5 стран по выручке:")
top_countries.show(truncate=False)

result_join_df = df.join(
    F.broadcast(top_countries.select("Country")),
    on="Country",
    how="inner"
)

stats = result_join_df.groupBy("Country").agg(
    F.count("*").alias("TransactionCount"),
    F.sum("Revenue").alias("TotalRevenue"),
    F.countDistinct("CustomerID").alias("UniqueCustomers")
).orderBy(F.desc("TotalRevenue"))

stats.show(truncate=False)

df_join_time = time.time() - start_time
print(f"\n⏱️  Время выполнения DataFrame с broadcast: {df_join_time:.3f} сек")
print(f"Обработано транзакций: {result_join_df.count():,}")

print("\n📊 План выполнения (обратите внимание на BroadcastHashJoin):")
stats.explain(mode="formatted")

КЕЙС 2: JOIN с broadcast - DataFrame
Топ-5 стран по выручке:
+--------+--------------------+
|Country |TotalRevenue        |
+--------+--------------------+
|Belgium |1.8658195065000024E8|
|Spain   |1.864707970500003E8 |
|Portugal|1.8627106749000013E8|
|Italy   |1.8594898179999995E8|
|Austria |1.8578270333999982E8|
+--------+--------------------+

+--------+----------------+--------------------+---------------+
|Country |TransactionCount|TotalRevenue        |UniqueCustomers|
+--------+----------------+--------------------+---------------+
|Belgium |289567          |1.8658195065000015E8|5942           |
|Spain   |288946          |1.8647079704999986E8|5942           |
|Portugal|288897          |1.8627106748999995E8|5942           |
|Italy   |288702          |1.8594898179999992E8|5942           |
|Austria |288194          |1.8578270334000006E8|5942           |
+--------+----------------+--------------------+---------------+


⏱️  Время выполнения DataFrame с broadcast: 0.955 сек
Обработан

In [7]:
print("=" * 80)
print("КЕЙС 2: JOIN без broadcast - RDD (для сравнения)")
print("=" * 80)

start_time = time.time()

rdd_country_revenue = df.rdd.map(lambda row: (row.Country, row.Revenue))
country_totals = rdd_country_revenue.reduceByKey(lambda a, b: a + b)
top_5_countries = country_totals.sortBy(lambda x: x[1], ascending=False).take(5)
top_5_set = {c[0] for c in top_5_countries}

print("Топ-5 стран по выручке:")
for i, (country, revenue) in enumerate(top_5_countries, 1):
    print(f"{i}. {country}: {revenue:.2f}")

rdd_full = df.rdd.map(lambda row: (row.Country, (row.CustomerID, row.Revenue)))

filtered_rdd = rdd_full.filter(lambda x: x[0] in top_5_set)

by_country = filtered_rdd.map(lambda x: (x[0], (1, x[1][1], x[1][0])))

country_stats = by_country.reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1], None)
).map(lambda x: (x[0], x[1][0], x[1][1]))

unique_customers = filtered_rdd.map(lambda x: (x[0], x[1][0])).distinct().countByKey()

result_rdd = []
for country, count, revenue in country_stats.collect():
    result_rdd.append((country, count, revenue, unique_customers.get(country, 0)))

result_rdd.sort(key=lambda x: x[2], reverse=True)

rdd_join_time = time.time() - start_time

print("\nСтатистика по топ-5 странам:")
for country, count, revenue, unique_cust in result_rdd:
    print(f"{country}: Transactions={count}, Revenue={revenue:.2f}, Customers={unique_cust}")

print(f"\n⏱️  Время выполнения RDD: {rdd_join_time:.3f} сек")
print(f"🚀 DataFrame быстрее в {rdd_join_time/df_join_time:.2f}x раз!")

КЕЙС 2: JOIN без broadcast - RDD (для сравнения)


Топ-5 стран по выручке:
1. Belgium: 186581950.65
2. Spain: 186470797.05
3. Portugal: 186271067.49
4. Italy: 185948981.80
5. Austria: 185782703.34


[Stage 55:>                                                       (0 + 14) / 14]


Статистика по топ-5 странам:
Belgium: Transactions=289567, Revenue=186581950.65, Customers=5942
Spain: Transactions=288946, Revenue=186470797.05, Customers=5942
Portugal: Transactions=288897, Revenue=186271067.49, Customers=5942
Italy: Transactions=288702, Revenue=185948981.80, Customers=5942
Austria: Transactions=288194, Revenue=185782703.34, Customers=5942

⏱️  Время выполнения RDD: 3.672 сек
🚀 DataFrame быстрее в 3.84x раз!


### Объяснение оптимизаций в Кейсе 2

**Catalyst Optimizer:**
- Автоматически определяет что топ-5 стран - маленькая таблица
- Применяет **Broadcast Hash Join** вместо Shuffle Hash Join
- Оптимизирует порядок операций (фильтр → join → агрегация)

**Broadcast Join:**
- Маленькая таблица (5 стран) рассылается всем executor'ам
- **НЕТ SHUFFLE** - огромная экономия на сетевых операциях
- Данные не записываются на диск
- JOIN происходит локально на каждом executor'е

**Tungsten:**
- Эффективное хранение данных при JOIN
- Бинарный формат без сериализации
- Hash таблицы оптимизированы для кэша процессора

**RDD недостатки:**
- Нужно вручную делать broadcast или терпеть shuffle
- filter() с Python set медленнее
- Множественные проходы: сначала топ-5, потом фильтр, потом агрегация
- distinct() для подсчета уникальных клиентов = еще один shuffle
- Нет автоматической оптимизации JOIN

## КЕЙС 3: Условная логика (when/otherwise)

**Задача:** Классифицировать заказы по размеру выручки:
- Small: Revenue < 50
- Medium: 50 <= Revenue < 200
- Large: Revenue >= 200

Затем посчитать статистику по каждой категории.

**Почему DataFrame быстрее:**
1. **Catalyst Optimizer** - оптимизирует условные выражения
2. **Codegen** - генерирует эффективный код для when/otherwise
3. **Predicate pushdown** - применяет фильтры на ранних стадиях
4. **Vectorization** - обрабатывает множество строк за раз

В RDD пришлось бы делать множественные filter + union, что приводит к:
- Нескольким проходам по данным
- Дублированию данных в памяти
- Неэффективному использованию ресурсов

In [8]:
print("=" * 80)
print("КЕЙС 3: Условная логика - DataFrame")
print("=" * 80)

start_time = time.time()

df_classified = df.withColumn(
    "OrderSize",
    F.when(F.col("Revenue") < 50, "Small")
     .when((F.col("Revenue") >= 50) & (F.col("Revenue") < 200), "Medium")
     .otherwise("Large")
)

result_conditional_df = df_classified.groupBy("OrderSize").agg(
    F.count("*").alias("OrderCount"),
    F.sum("Revenue").alias("TotalRevenue"),
    F.avg("Revenue").alias("AvgRevenue"),
    F.min("Revenue").alias("MinRevenue"),
    F.max("Revenue").alias("MaxRevenue")
).orderBy("OrderSize")

result_conditional_df.show(truncate=False)

df_conditional_time = time.time() - start_time
print(f"\n⏱️  Время выполнения DataFrame: {df_conditional_time:.3f} сек")

print("\n📊 План выполнения:")
result_conditional_df.explain(mode="formatted")

КЕЙС 3: Условная логика - DataFrame
+---------+----------+--------------------+------------------+----------+----------+
|OrderSize|OrderCount|TotalRevenue        |AvgRevenue        |MinRevenue|MaxRevenue|
+---------+----------+--------------------+------------------+----------+----------+
|Large    |2754089   |2.3230283294000015E9|843.4833912048599 |200.0     |2500.0    |
|Medium   |693519    |8.349963524000002E7 |120.39992450098703|50.0      |199.99    |
|Small    |302457    |8265585.600000009   |27.328134577807784|0.5       |49.99     |
+---------+----------+--------------------+------------------+----------+----------+


⏱️  Время выполнения DataFrame: 0.187 сек

📊 План выполнения:
== Physical Plan ==
AdaptiveSparkPlan (12)
+- Sort (11)
   +- Exchange (10)
      +- HashAggregate (9)
         +- Exchange (8)
            +- HashAggregate (7)
               +- Project (6)
                  +- InMemoryTableScan (1)
                        +- InMemoryRelation (2)
                       

In [9]:
print("=" * 80)
print("КЕЙС 3: Условная логика - RDD (для сравнения)")
print("=" * 80)

start_time = time.time()

rdd_base = df.rdd.map(lambda row: row.Revenue)

small_rdd = rdd_base.filter(lambda rev: rev < 50).map(lambda rev: ("Small", rev))
medium_rdd = rdd_base.filter(lambda rev: 50 <= rev < 200).map(lambda rev: ("Medium", rev))
large_rdd = rdd_base.filter(lambda rev: rev >= 200).map(lambda rev: ("Large", rev))

combined_rdd = small_rdd.union(medium_rdd).union(large_rdd)

def aggregate_stats(revenues):
    rev_list = list(revenues)
    count = len(rev_list)
    total = sum(rev_list)
    avg = total / count if count > 0 else 0
    min_val = min(rev_list) if rev_list else 0
    max_val = max(rev_list) if rev_list else 0
    return (count, total, avg, min_val, max_val)

result_rdd = combined_rdd.groupByKey().mapValues(aggregate_stats).collect()

rdd_conditional_time = time.time() - start_time

print("\nСтатистика по размерам заказов:")
for order_size, (count, total, avg, min_val, max_val) in sorted(result_rdd):
    print(f"{order_size}:")
    print(f"  Count: {count}")
    print(f"  Total Revenue: {total:.2f}")
    print(f"  Avg Revenue: {avg:.2f}")
    print(f"  Min Revenue: {min_val:.2f}")
    print(f"  Max Revenue: {max_val:.2f}")

print(f"\n⏱️  Время выполнения RDD: {rdd_conditional_time:.3f} сек")
print(f"🚀 DataFrame быстрее в {rdd_conditional_time/df_conditional_time:.2f}x раз!")

КЕЙС 3: Условная логика - RDD (для сравнения)


[Stage 60:======================================================> (41 + 1) / 42]


Статистика по размерам заказов:
Large:
  Count: 2754089
  Total Revenue: 2323028329.40
  Avg Revenue: 843.48
  Min Revenue: 200.00
  Max Revenue: 2500.00
Medium:
  Count: 693519
  Total Revenue: 83499635.24
  Avg Revenue: 120.40
  Min Revenue: 50.00
  Max Revenue: 199.99
Small:
  Count: 302457
  Total Revenue: 8265585.60
  Avg Revenue: 27.33
  Min Revenue: 0.50
  Max Revenue: 49.99

⏱️  Время выполнения RDD: 3.154 сек
🚀 DataFrame быстрее в 16.86x раз!


### Объяснение оптимизаций в Кейсе 3

**Catalyst Optimizer:**
- Преобразует when/otherwise в эффективные условные выражения
- Применяет constant folding для упрощения условий
- Оптимизирует порядок проверки условий

**Code Generation:**
- Генерирует специализированный байт-код для условной логики
- Избегает виртуальных вызовов методов
- Использует branch prediction процессора

**Один проход по данным:**
- DataFrame обрабатывает все условия за один проход
- Нет дублирования данных в памяти
- Эффективное использование кэша процессора

**RDD недостатки:**
- Три отдельных filter операции = три прохода по данным
- union() создает копии данных
- groupByKey() собирает все значения в память (опасно!)
- Нет автоматической оптимизации условий

## (*) Дополнительное задание: SQL vs DataFrame API

Найдем 2 кейса, где SQL-запрос выполняется быстрее эквивалентного DataFrame API.

### Подготовка: регистрация временной таблицы

In [10]:
df.createOrReplaceTempView("sales")

### SQL vs DataFrame - Кейс 1: Сложные агрегации с HAVING

**Задача:** Найти страны с общей выручкой > 100,000 и количеством транзакций > 1,000

In [11]:
print("SQL подход:")
start_time = time.time()

sql_result = spark.sql("""
    SELECT 
        Country,
        SUM(Revenue) as TotalRevenue,
        COUNT(*) as TransactionCount
    FROM sales
    GROUP BY Country
    HAVING SUM(Revenue) > 100000 AND COUNT(*) > 1000
    ORDER BY TotalRevenue DESC
""")

sql_result.show(10, truncate=False)
sql_time_1 = time.time() - start_time

print(f"⏱️  SQL время: {sql_time_1:.3f} сек")
print("\n📊 SQL план:")
sql_result.explain(mode="formatted")

SQL подход:
+--------------+--------------------+----------------+
|Country       |TotalRevenue        |TransactionCount|
+--------------+--------------------+----------------+
|Belgium       |1.8658195065000024E8|289567          |
|Spain         |1.864707970500003E8 |288946          |
|Portugal      |1.8627106749000013E8|288897          |
|Italy         |1.8594898179999995E8|288702          |
|Austria       |1.8578270333999982E8|288194          |
|France        |1.8572960912999973E8|288737          |
|Denmark       |1.8563956083000004E8|287715          |
|United Kingdom|1.8562559637000015E8|288585          |
|Switzerland   |1.8557651045999992E8|288372          |
|Netherlands   |1.8550434239000005E8|288450          |
+--------------+--------------------+----------------+
only showing top 10 rows
⏱️  SQL время: 0.186 сек

📊 SQL план:
== Physical Plan ==
AdaptiveSparkPlan (12)
+- Sort (11)
   +- Exchange (10)
      +- Filter (9)
         +- HashAggregate (8)
            +- Exchange (7)
 

In [12]:
print("DataFrame API подход:")
start_time = time.time()

df_result = df.groupBy("Country").agg(
    F.sum("Revenue").alias("TotalRevenue"),
    F.count("*").alias("TransactionCount")
).filter(
    (F.col("TotalRevenue") > 100000) & (F.col("TransactionCount") > 1000)
).orderBy(F.desc("TotalRevenue"))

df_result.show(10, truncate=False)
df_api_time_1 = time.time() - start_time

print(f"⏱️  DataFrame API время: {df_api_time_1:.3f} сек")
print("\n📊 DataFrame API план:")
df_result.explain(mode="formatted")

if df_api_time_1 > sql_time_1:
    print(f"\n⚡ SQL быстрее на {((df_api_time_1 - sql_time_1) / sql_time_1 * 100):.1f}%")
else:
    print(f"\n⚡ DataFrame API быстрее на {((sql_time_1 - df_api_time_1) / df_api_time_1 * 100):.1f}%")

DataFrame API подход:
+--------------+--------------------+----------------+
|Country       |TotalRevenue        |TransactionCount|
+--------------+--------------------+----------------+
|Belgium       |1.8658195065000024E8|289567          |
|Spain         |1.864707970500003E8 |288946          |
|Portugal      |1.8627106749000013E8|288897          |
|Italy         |1.8594898179999995E8|288702          |
|Austria       |1.8578270333999982E8|288194          |
|France        |1.8572960912999973E8|288737          |
|Denmark       |1.8563956083000004E8|287715          |
|United Kingdom|1.8562559637000015E8|288585          |
|Switzerland   |1.8557651045999992E8|288372          |
|Netherlands   |1.8550434239000005E8|288450          |
+--------------+--------------------+----------------+
only showing top 10 rows
⏱️  DataFrame API время: 0.148 сек

📊 DataFrame API план:
== Physical Plan ==
AdaptiveSparkPlan (12)
+- Sort (11)
   +- Exchange (10)
      +- Filter (9)
         +- HashAggregate (8)

**Объяснение разницы:**

В данном кейсе DataFrame API оказался быстрее SQL! Это происходит потому что:

1. **Кэширование данных:**
   - DataFrame использует уже закэшированные данные эффективнее
   - SQL может пересканировать данные

2. **Оптимизация фильтра:**
   - DataFrame API применяет filter() сразу после агрегации
   - Catalyst оптимизирует это в единый план

3. **Меньше парсинга:**
   - DataFrame API не требует парсинга SQL строки
   - Прямое построение плана выполнения

**Вывод:** На небольших данных с кэшированием разница минимальна. SQL показывает преимущество на больших данных без кэша или при сложных подзапросах.

### SQL vs DataFrame - Кейс 2: Сложные JOIN с подзапросами

**Задача:** Найти товары, которые продавались в странах с общей выручкой > 50,000

In [13]:
print("SQL подход с подзапросом:")
start_time = time.time()

sql_result_2 = spark.sql("""
    SELECT DISTINCT
        s.StockCode,
        s.Description,
        s.Country
    FROM sales s
    WHERE s.Country IN (
        SELECT Country
        FROM sales
        GROUP BY Country
        HAVING SUM(Revenue) > 50000
    )
    ORDER BY s.Country, s.StockCode
""")

sql_result_2.show(20, truncate=False)
sql_time_2 = time.time() - start_time

print(f"⏱️  SQL время: {sql_time_2:.3f} сек")
print(f"Количество уникальных комбинаций: {sql_result_2.count()}")
print("\n📊 SQL план:")
sql_result_2.explain(mode="formatted")

SQL подход с подзапросом:
+---------+-------------------------------+-------+
|StockCode|Description                    |Country|
+---------+-------------------------------+-------+
|10002    |INFLATABLE POLITICAL GLOBE     |Austria|
|10120    |DOGGY RUBBER                   |Austria|
|10123C   |HEARTS WRAPPING TAPE           |Austria|
|10125    |MINI FUNKY DESIGN CAKE CASES   |Austria|
|10133    |COLOURING PENCILS BROWN TUBE   |Austria|
|10135    |COLOURING PENCILS TUBE SKULLS  |Austria|
|11001    |ASSTD DESIGN RACING CARS       |Austria|
|15030    |ASSORTED COLOURS SILK FAN      |Austria|
|15034    |BLUE POLKADOT WRAP             |Austria|
|15036    |ASSORTED COLOUR BIRD ORNAMENT  |Austria|
|16014    |SMALL CHINESE STYLE SCISSOR    |Austria|
|17003    |BROCADE RING PURSE             |Austria|
|20665    |RED RETROSPOT CHARLOTTE BAG    |Austria|
|20719    |WOODLAND CHARLOTTE BAG         |Austria|
|20725    |LUNCH BAG RED RETROSPOT        |Austria|
|20727    |LUNCH BAG BLACK SKULL      

In [14]:
print("DataFrame API подход:")
start_time = time.time()

high_revenue_countries = df.groupBy("Country").agg(
    F.sum("Revenue").alias("TotalRevenue")
).filter(F.col("TotalRevenue") > 50000).select("Country")

df_result_2 = df.join(
    high_revenue_countries,
    on="Country",
    how="inner"
).select("StockCode", "Description", "Country").distinct().orderBy("Country", "StockCode")

df_result_2.show(20, truncate=False)
df_api_time_2 = time.time() - start_time

print(f"⏱️  DataFrame API время: {df_api_time_2:.3f} сек")
print(f"Количество уникальных комбинаций: {df_result_2.count()}")
print("\n📊 DataFrame API план:")
df_result_2.explain(mode="formatted")

if df_api_time_2 > sql_time_2:
    print(f"\n⚡ SQL быстрее на {((df_api_time_2 - sql_time_2) / sql_time_2 * 100):.1f}%")
else:
    print(f"\n⚡ DataFrame API быстрее на {((sql_time_2 - df_api_time_2) / df_api_time_2 * 100):.1f}%")

DataFrame API подход:
+---------+-------------------------------+-------+
|StockCode|Description                    |Country|
+---------+-------------------------------+-------+
|10002    |INFLATABLE POLITICAL GLOBE     |Austria|
|10120    |DOGGY RUBBER                   |Austria|
|10123C   |HEARTS WRAPPING TAPE           |Austria|
|10125    |MINI FUNKY DESIGN CAKE CASES   |Austria|
|10133    |COLOURING PENCILS BROWN TUBE   |Austria|
|10135    |COLOURING PENCILS TUBE SKULLS  |Austria|
|11001    |ASSTD DESIGN RACING CARS       |Austria|
|15030    |ASSORTED COLOURS SILK FAN      |Austria|
|15034    |BLUE POLKADOT WRAP             |Austria|
|15036    |ASSORTED COLOUR BIRD ORNAMENT  |Austria|
|16014    |SMALL CHINESE STYLE SCISSOR    |Austria|
|17003    |BROCADE RING PURSE             |Austria|
|20665    |RED RETROSPOT CHARLOTTE BAG    |Austria|
|20719    |WOODLAND CHARLOTTE BAG         |Austria|
|20725    |LUNCH BAG RED RETROSPOT        |Austria|
|20727    |LUNCH BAG BLACK SKULL          

**Объяснение разницы в Кейсе 2:**

**Почему SQL быстрее:**

1. **Оптимизация подзапросов:**
   - SQL оптимизатор может преобразовать IN подзапрос в semi-join
   - Это более эффективно, чем явный inner join
   - Catalyst лучше оптимизирует декларативные SQL конструкции

2. **Broadcast join:**
   - Список стран с высокой выручкой маленький
   - SQL автоматически применяет broadcast join
   - DataFrame API может не распознать эту возможность

3. **Predicate pushdown:**
   - SQL может применить фильтр HAVING до JOIN
   - В DataFrame API фильтр применяется после создания промежуточного результата

4. **Меньше промежуточных этапов:**
   - SQL создает более компактный план выполнения
   - DataFrame API создает дополнительные этапы для join и distinct

## Сводная таблица производительности

In [15]:
import pandas as pd

performance_data = {
    "Кейс": [
        "1. Множественные агрегации",
        "2. JOIN с broadcast оптимизацией",
        "3. Условная логика",
        "SQL vs DF: Агрегации с HAVING",
        "SQL vs DF: JOIN с подзапросами"
    ],
    "DataFrame (сек)": [
        f"{df_time:.3f}",
        f"{df_window_time:.3f}",
        f"{df_conditional_time:.3f}",
        f"{df_api_time_1:.3f}",
        f"{df_api_time_2:.3f}"
    ],
    "RDD/Альтернатива (сек)": [
        f"{rdd_time:.3f}",
        f"{rdd_window_time:.3f}",
        f"{rdd_conditional_time:.3f}",
        f"{sql_time_1:.3f}",
        f"{sql_time_2:.3f}"
    ],
    "Ускорение": [
        f"{rdd_time/df_time:.2f}x",
        f"{rdd_complex_time/df_complex_time:.2f}x",
        f"{rdd_conditional_time/df_conditional_time:.2f}x",
        f"DF быстрее на {((sql_time_1 - df_api_time_1) / df_api_time_1 * 100):.1f}%" if df_api_time_1 < sql_time_1 else f"SQL быстрее на {((df_api_time_1 - sql_time_1) / sql_time_1 * 100):.1f}%",
        f"DF быстрее на {((sql_time_2 - df_api_time_2) / df_api_time_2 * 100):.1f}%" if df_api_time_2 < sql_time_2 else f"SQL быстрее на {((df_api_time_2 - sql_time_2) / sql_time_2 * 100):.1f}%"
    ]
}

perf_df = pd.DataFrame(performance_data)
print("\n" + "=" * 100)
print("СВОДНАЯ ТАБЛИЦА ПРОИЗВОДИТЕЛЬНОСТИ")
print("=" * 100)
print(perf_df.to_string(index=False))
print("=" * 100)

NameError: name 'df_window_time' is not defined

In [ ]:
print("DataFrame API подход:")
start_time = time.time()

high_revenue_countries = df.groupBy("Country").agg(
    F.sum("Revenue").alias("TotalRevenue")
).filter(F.col("TotalRevenue") > 50000).select("Country")

df_result_2 = df.join(
    high_revenue_countries,
    on="Country",
    how="inner"
).select("StockCode", "Description", "Country").distinct().orderBy("Country", "StockCode")

df_result_2.show(20, truncate=False)
df_api_time_2 = time.time() - start_time

print(f"⏱️  DataFrame API время: {df_api_time_2:.3f} сек")
print(f"Количество уникальных комбинаций: {df_result_2.count()}")
print("\n📊 DataFrame API план:")
df_result_2.explain(mode="formatted")

if df_api_time_2 > sql_time_2:
    print(f"\n⚡ SQL быстрее на {((df_api_time_2 - sql_time_2) / sql_time_2 * 100):.1f}%")
else:
    print(f"\n⚡ DataFrame API быстрее на {((sql_time_2 - df_api_time_2) / df_api_time_2 * 100):.1f}%")

**Объяснение разницы в Кейсе 2:**

В этом кейсе результаты практически идентичны. Это показывает:

1. **Планы выполнения похожи:**
   - SQL использует LeftSemi join (из плана)
   - DataFrame API использует Inner join
   - Catalyst оптимизирует оба подхода похоже

2. **Влияние кэша:**
   - Оба используют закэшированные данные
   - На небольших объемах разница минимальна

3. **Когда SQL реально быстрее:**
   - На больших данных (5-10M+ строк)
   - При множественных вложенных подзапросах
   - Без кэширования данных

**Вывод:** На текущем датасете (750K строк) с кэшем разница в пределах погрешности. SQL показывает преимущество на больших данных или сложных запросах.

## Выводы

### 1. DataFrame vs RDD - когда DataFrame побеждает:

**Множественные агрегации:**
- DataFrame объединяет все агрегации в один проход
- Tungsten обеспечивает эффективную работу с памятью
- Whole-stage codegen генерирует оптимизированный код
- RDD требует несколько проходов и ручного объединения результатов

**JOIN с broadcast оптимизацией:**
- Catalyst автоматически применяет broadcast join
- Нет shuffle - данные не передаются по сети
- Tungsten эффективно работает с hash таблицами
- RDD требует вручную делать broadcast или терпеть shuffle

**Условная логика:**
- when/otherwise компилируется в эффективный код
- Один проход по данным вместо нескольких filter + union
- Нет дублирования данных в памяти
- Catalyst оптимизирует порядок проверки условий

### 2. SQL vs DataFrame API - результаты:

**На текущем датасете (750K строк):**
- DataFrame API оказался быстрее или одинаково с SQL
- Причина: данные закэшированы, объем небольшой для локального режима

**Когда SQL реально быстрее:**
- На больших данных (5-10M+ строк) без кэша
- При сложных вложенных подзапросах (3+ уровня)
- При множественных JOIN с разными таблицами
- Когда SQL оптимизатор может применить broadcast автоматически

**Вывод:** Для демонстрации преимущества SQL нужны большие данные или более сложные запросы. На учебных датасетах разница минимальна.

### 3. Общие рекомендации:

1. **Используйте DataFrame вместо RDD** для табличных данных и стандартных операций
2. **Предпочитайте SQL** для сложных запросов с подзапросами и множественными JOIN
3. **Всегда проверяйте план** через explain() - он покажет реальные оптимизации
4. **Кэшируйте данные** если они используются многократно
5. **Избегайте UDF** - они ломают оптимизации Catalyst
6. **Используйте встроенные функции** - их >200 в pyspark.sql.functions
7. **Для RDD** спускайтесь только когда данные не табличные или нужна специфическая логика

### 4. Ключевые оптимизации Catalyst/Tungsten:

- **Catalyst Optimizer:** логическая и физическая оптимизация планов
- **Tungsten:** бинарный формат, off-heap память, code generation
- **Predicate pushdown:** фильтры применяются как можно раньше
- **Broadcast join:** маленькие таблицы рассылаются всем executor'ам
- **Whole-stage codegen:** генерация оптимизированного байт-кода

In [ ]:
spark.stop()
print("✅ Spark сессия завершена")